# 05. Design a cut and count
**After Lectures 5–7 · Draft teaching exercise; use instructor-approved samples.**

First use the UI to inspect signal and background and design your cuts. Copy the final settings below, or export the current analysis notebook. Freeze the selection before the instructor releases pseudo-data. Independent Monte Carlo is not itself experimental pseudo-data: the instructor must normalize and Poisson-sample the event count.

Save your own copy before editing. Run cells from the top. Empty sample selections deliberately do nothing; choose IDs from the available-samples table.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from findingz.catalog import load_catalog
from findingz.delphes import default_run_root
from findingz.hypotheses import build_sample_library

library = build_sample_library(load_catalog(), default_run_root())
display(pd.DataFrame([{"sample_id": k, "label": s.label, "cross_section_pb": s.cross_section_pb,
                      "events": s.generated_events, "config": s.config} for k, s in library.items()]))
def get_sample(sample_id):
    if sample_id not in library:
        raise ValueError(f"Sample {sample_id!r} is unavailable. Choose an ID from the table above.")
    return library[sample_id]
def weights(frame):
    return pd.to_numeric(frame.get("weight", pd.Series(1., index=frame.index)))


In [ ]:
from findingz.hypotheses import validate_counting_samples
from findingz.counting import summarize_cut_and_count
signal_id = None
background_ids = []
luminosity_fb = 1.
mass_low, mass_high = 80., 100.
pt_min, eta_max = 20., 2.4
background_uncertainty = 0.10
pseudo_data_path = None  # released JSON with "events": list of event dictionaries
def select(frame):
    return frame.loc[(frame.mll>=mass_low)&(frame.mll<=mass_high)
        &(frame.l1_pt>pt_min)&(frame.l2_pt>pt_min)
        &(abs(frame.l1_eta)<eta_max)&(abs(frame.l2_eta)<eta_max)]
if signal_id and background_ids:
    samples = [get_sample(key) for key in [signal_id,*background_ids]]
    validate_counting_samples(samples)
    signal = select(samples[0].expected_frame(luminosity_fb))
    background = pd.concat([select(s.expected_frame(luminosity_fb)) for s in samples[1:]],ignore_index=True)
    observed = None
    if pseudo_data_path:
        observed = select(pd.DataFrame(json.loads(Path(pseudo_data_path).read_text())["events"], columns=background.columns))
    result = summarize_cut_and_count(signal,background,observed,
                                    background_uncertainty_fraction=background_uncertainty)
    display(pd.Series(vars(result)))
    frozen_selection = dict(signal=signal_id,backgrounds=background_ids,luminosity_fb=luminosity_fb,
        mass_window=[mass_low,mass_high],pt_min=pt_min,eta_max=eta_max,
        background_uncertainty=background_uncertainty)
    display(frozen_selection)

## Questions and submission
1. Report yields after each cut and defend the selection before opening pseudo-data.
2. Compare statistical and background-systematic limitations.
3. Interpret the expected limit as a teaching approximation, not CLs.
4. For observed data, note that the helper's Poisson p-value treats the background mean as known; the uncertainty slider is not included in that p-value.
5. Submit your frozen settings before data release and distinguish discovery evidence from an exclusion limit.

Submit your edited notebook with figures, units, sample IDs, generation settings, and a short interpretation. Do not equate Monte Carlo event count with experimental luminosity.